In [1]:
# Load the Movies DB into a separate in-notebook sqlite connection (instructor-only)
import os, re, sqlite3
import pandas as pd

DATA_DIR = '/content/data'   # adjust if your folder lives elsewhere

mdb = sqlite3.connect(':memory:')

def mq(sql):
    '''Run a query against the Movies DB and return a DataFrame.'''
    return pd.read_sql_query(sql, mdb)

# Postgres -> SQLite tweaks applied as we read each .sql file
for f in sorted(os.listdir(DATA_DIR)):
    if not f.endswith('.sql'):
        continue
    with open(os.path.join(DATA_DIR, f)) as fh:
        sql = fh.read()
    sql = re.sub(r'DROP SCHEMA.*?CASCADE;', '', sql, flags=re.IGNORECASE | re.DOTALL)
    sql = re.sub(r'CREATE SCHEMA\s+\w+\s*;', '', sql, flags=re.IGNORECASE)
    sql = sql.replace('movies.', '')
    sql = re.sub(r'INT NOT NULL GENERATED BY DEFAULT AS IDENTITY', 'INTEGER', sql)
    sql = re.sub(r'varchar\(\d+\)', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'decimal\(\d+,\d+\)', 'REAL', sql, flags=re.IGNORECASE)
    sql = re.sub(r'BIGINT', 'INTEGER', sql)
    sql = re.sub(r'date\s+DEFAULT', 'TEXT DEFAULT', sql)
    try:
        mdb.executescript(sql)
    except Exception:
        # Some files emit harmless 'no transaction' warnings after auto-commit; ignore.
        pass

# Sanity check
print('Movies DB loaded. Row counts:')
for t in ['movie', 'genre', 'person', 'production_company', 'keyword',
          'movie_genres', 'movie_cast', 'movie_crew', 'movie_company', 'movie_keywords']:
    try:
        n = mdb.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'  {t:22s}: {n:>8,} rows')
    except Exception:
        print(f'  {t:22s}: MISSING')

Movies DB loaded. Row counts:
  movie                 :    4,803 rows
  genre                 :       20 rows
  person                :  104,842 rows
  production_company    :    5,047 rows
  keyword               :    9,794 rows
  movie_genres          :   12,160 rows
  movie_cast            :  106,257 rows
  movie_crew            :  129,581 rows
  movie_company         :   13,677 rows
  movie_keywords        :   36,162 rows


In [2]:
# Task 1: Calculate the Average Budget Growth Rate for Each Production Company
# Calculate the average budget growth rate for each production company across all movies they have produced. Use window functions to determine the budget growth rate and then calculate the average growth rate.




mq('''WITH company_movies AS (
    SELECT
        pc.company_name,
        m.title,
        m.release_date,
        m.budget,
        LAG(m.budget) OVER (
            PARTITION BY pc.company_id
            ORDER BY m.release_date
        ) AS previous_budget
    FROM movie m
    JOIN movie_company mc
        ON mc.movie_id = m.movie_id
    JOIN production_company pc
        ON pc.company_id = mc.company_id
    WHERE m.budget > 0
),
budget_growth AS (
    SELECT
        company_name,
        title,
        budget,
        previous_budget,
        1.0 * (budget - previous_budget) / previous_budget AS growth_rate
    FROM company_movies
    WHERE previous_budget IS NOT NULL
      AND previous_budget > 0
)
SELECT
    company_name,
    ROUND(AVG(growth_rate), 3) AS avg_budget_growth_rate
FROM budget_growth
GROUP BY company_name
ORDER BY avg_budget_growth_rate DESC;''')



,company_name,avg_budget_growth_rate
0,Artists Production Group (APG),1428570.585
1,Focus Films,535713.452
2,Homegrown Pictures,466665.667
3,Beijing New Picture Film Co. Ltd.,427271.727
4,Muse Productions,250000.335
...,...,...
1246,Arte France,-0.972
1247,The Steve Tisch Company,-0.979
1248,Warp Films,-0.982
1249,Bu00f3rd Scannu00e1n na hu00c9ireann,-0.985


In [3]:


# Task 2: Determine the Most Consistently High-Rated Actor
# Identify the actor who has appeared in the most movies that are rated above the average rating of all movies. Use window functions and CTEs to calculate the average rating and filter the actors based on this criterion.
mq('''

WITH movie_avg AS (
    SELECT
        AVG(vote_average) AS avg_rating
    FROM movie
),
above_average_movies AS (
    SELECT
        m.movie_id,
        m.title,
        m.vote_average
    FROM movie m
    JOIN movie_avg ma
    WHERE m.vote_average > ma.avg_rating
),
actor_counts AS (
    SELECT
        p.person_name,
        COUNT(*) AS above_avg_movie_count,
        RANK() OVER (
            ORDER BY COUNT(*) DESC
        ) AS rnk
    FROM above_average_movies aam
    JOIN movie_cast mc
        ON mc.movie_id = aam.movie_id
    JOIN person p
        ON p.person_id = mc.person_id
    GROUP BY p.person_id, p.person_name
)
SELECT
    person_name,
    above_avg_movie_count
FROM actor_counts
WHERE rnk = 1;
'''
)


,person_name,above_avg_movie_count
0,Samuel L. Jackson,45


In [4]:


# Task 3: Calculate the Rolling Average Revenue for Each Genre
# Calculate the rolling average revenue for movies within each genre, considering only the last three movies released in the genre. Use window functions with the ROWS frame specification to achieve this.

mq('''
SELECT
    g.genre_name,
    m.title,
    m.release_date,
    m.revenue,
    ROUND(
        AVG(m.revenue) OVER (
            PARTITION BY g.genre_id
            ORDER BY m.release_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS rolling_avg_revenue_last_3
FROM movie m
JOIN movie_genres mg
    ON mg.movie_id = m.movie_id
JOIN genre g
    ON g.genre_id = mg.genre_id
WHERE m.revenue > 0
ORDER BY g.genre_name, m.release_date;

''')


,genre_name,title,release_date,revenue,rolling_avg_revenue_last_3
0,Action,Hell's Angels,1930-11-15,8000000,8.000000e+06
1,Action,The Charge of the Light Brigade,1936-10-20,2736000,5.368000e+06
2,Action,Sands of Iwo Jima,1949-12-14,7800000,6.178667e+06
3,Action,Annie Get Your Gun,1950-05-17,8000000,6.178667e+06
4,Action,The Greatest Show on Earth,1952-01-10,36000000,1.726667e+07
...,...,...,...,...,...
8893,Western,The Homesman,2014-05-18,3442853,1.727003e+08
8894,Western,Slow West,2015-04-16,229094,3.098729e+07
8895,Western,The Hateful Eight,2015-12-25,155760117,5.314402e+07
8896,Western,The Revenant,2015-12-25,532950503,2.296466e+08


In [5]:
# Task 4: Identify the Highest-Grossing Movie Series
# Identify the movie series (based on shared keywords) with the highest total revenue. Use window functions and CTEs to group movies by their series and calculate the total revenue.


mq('''

WITH keyword_revenue AS (
    SELECT
        k.keyword_name,
        m.title,
        m.revenue,
        SUM(m.revenue) OVER (
            PARTITION BY k.keyword_id
        ) AS total_keyword_revenue,
        COUNT(*) OVER (
            PARTITION BY k.keyword_id
        ) AS movies_in_keyword
    FROM movie m
    JOIN movie_keywords mk
        ON mk.movie_id = m.movie_id
    JOIN keyword k
        ON k.keyword_id = mk.keyword_id
    WHERE m.revenue > 0
),
ranked_keywords AS (
    SELECT DISTINCT
        keyword_name,
        total_keyword_revenue,
        movies_in_keyword,
        RANK() OVER (
            ORDER BY total_keyword_revenue DESC
        ) AS rnk
    FROM keyword_revenue
    WHERE movies_in_keyword > 1
)
SELECT
    keyword_name,
    total_keyword_revenue,
    movies_in_keyword
FROM ranked_keywords
WHERE rnk = 1;

''')

,keyword_name,total_keyword_revenue,movies_in_keyword
0,duringcreditsstinger,57827617707,278
